In [20]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import rasterio
from rasterstats import zonal_stats

# -------------------------------------------------------------------
# 1. Rutas
# -------------------------------------------------------------------

FP_SIM = Path("../data/processed/censo/manzanas_penalolen_sim.parquet")
FP_NDVI = Path("../data/raw/ndvi_penalolen.tif")

# -------------------------------------------------------------------
# 2. Leer tabla ya lista desde script 2
# -------------------------------------------------------------------

manz_sim = gpd.read_parquet(FP_SIM)

print("Dimensión base:", manz_sim.shape)
print("CRS base:", manz_sim.crs)

# -------------------------------------------------------------------
# 3. Abrir raster NDVI
# -------------------------------------------------------------------

src = rasterio.open(FP_NDVI)
print("CRS raster:", src.crs)

# -------------------------------------------------------------------
# 4. Reproyectar temporalmente al CRS del raster para zonal stats
# -------------------------------------------------------------------

manz_sim_ndvi = manz_sim.to_crs(src.crs)

# -------------------------------------------------------------------
# 5. Calcular estadísticas zonales
# -------------------------------------------------------------------

zs = zonal_stats(
    manz_sim_ndvi,
    FP_NDVI,
    stats=["mean", "median", "max"],
    nodata=None
)

zs_df = pd.DataFrame(zs)

# -------------------------------------------------------------------
# 6. Agregar NDVI a la tabla original
# -------------------------------------------------------------------

manz_sim["ndvi_mean"] = zs_df["mean"].values
manz_sim["ndvi_median"] = zs_df["median"].values
manz_sim["ndvi_max"] = zs_df["max"].values

# Normalización 0-1 para simulación
manz_sim["ndvi_norm"] = (
    manz_sim["ndvi_mean"] - manz_sim["ndvi_mean"].min()
) / (
    manz_sim["ndvi_mean"].max() - manz_sim["ndvi_mean"].min()
)

# Opcional: quintiles para mapas / revisión
manz_sim["ndvi_q"] = pd.qcut(
    manz_sim["ndvi_mean"],
    5,
    labels=["Muy bajo", "Bajo", "Medio", "Alto", "Muy alto"]
)

# -------------------------------------------------------------------
# 7. Chequeos rápidos
# -------------------------------------------------------------------

print(manz_sim[["ndvi_mean", "ndvi_median", "ndvi_max", "ndvi_norm"]].describe())
print(manz_sim[["ndvi_mean", "ndvi_median", "ndvi_max", "ndvi_norm"]].isna().mean())

# -------------------------------------------------------------------
# 8. Sobrescribir archivo final
# -------------------------------------------------------------------

manz_sim_export = manz_sim.copy()

for col in manz_sim_export.select_dtypes(include="category").columns:
    manz_sim_export[col] = manz_sim_export[col].astype(str)

manz_sim_export.to_parquet(FP_SIM)

manz_sim_export.to_file(
    FP_SIM.with_suffix(".gpkg"),
    layer="manzanas_sim",
    driver="GPKG"
)

print("Archivo actualizado con NDVI en:", FP_SIM.resolve())

Dimensión base: (1629, 9)
CRS base: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "WGS 84 / UTM zone 19S", "base_crs": {"name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": 